# Week 3 — AG News + MiniLM Quantum Kernel (N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — Encoder Comparison

**This is the MINILM version of the AGNEWS quantum kernel run.**
Same structure as your BERT Week 3 — loads `agnews_minilm_embeddings.npz`, uses macro F1.

**Inputs:** `agnews_minilm_embeddings.npz`, `agnews_minilm_baseline_results.pkl` (from Week 2)
**Runtime:** ~83 min (1000x1000 kernel matrix)

In [ ]:
!pip install pennylane pennylane-lightning --quiet
import pennylane as qml
import numpy as np
import time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__}")
DEVICE = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage + helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
TAG = 'agnews_minilm'
print(f"Save dir: {SAVE_DIR} | Tag: {TAG}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load AG News + MiniLM embeddings

In [ ]:
data = np.load(_find(f'{TAG}_embeddings.npz'))
X_tr = data['X_tr_subset'].astype(np.float64)
y_tr = data['y_tr_subset']
X_te = data['X_te_subset'].astype(np.float64)
y_te = data['y_te_subset']

with open(_find(f'{TAG}_baseline_results.pkl'), 'rb') as f:
    baselines = pickle.load(f)

print(f"Train {X_tr.shape}  Test {X_te.shape}")
print(f"Range [{X_tr.min():.4f}, {X_tr.max():.4f}]  (must be [0, pi])")
assert X_tr.shape == (1000, 8), "Expected (1000,8) — re-run Week 2 for this config"
assert X_te.shape == (200, 8)
print("N=1000 confirmed.")
print(f"\nBaselines (metric=macro F1):")
for k, v in baselines.items():
    print(f"  {k:<12} acc={v['accuracy']:.4f}  f1={v['f1']:.4f}")

---
## Step 2: IQP quantum kernel (identical circuit to all other Week 3s)

In [ ]:
N_QUBITS = 8

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(DEVICE)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])

assert np.isclose(quantum_kernel(X_tr[0], X_tr[0]), 1.0, atol=1e-5)
t0 = time.time()
for _ in range(3): quantum_kernel(X_tr[0], X_tr[1])
ms = (time.time()-t0)/3*1000
print(f"Sanity OK. {ms:.1f}ms/eval | N=1000 est ~{1000*1000*ms/1000/60:.0f} min")

---
## Step 3: Compute kernel matrix — saved immediately after (~83 min)

In [ ]:
print(f"Computing {TAG} kernel matrix (N=1000)...")
print("Saving immediately when done.")
t0 = time.time()
K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min")

print("\nSaving kernel matrices immediately:")
checkpoint(f'{TAG}_kernel_matrices.npz',
           lambda p: np.savez(p, K_train_quantum=K_train, K_test_quantum=K_test,
                              X_tr=X_tr, y_tr=y_tr))
print("Safe.")

---
## Step 4: Quantum SVM + comparison

In [ ]:
qsvm = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr)
y_pred = qsvm.predict(K_test)
acc_q = float(accuracy_score(y_te, y_pred))
f1_q  = float(f1_score(y_te, y_pred, average='macro'))

print("="*50)
print(f"QUANTUM KERNEL SVM ({TAG}, N=1000, 8 qubits):")
print(f"  Accuracy : {acc_q:.4f}")
print(f"  Macro F1: {f1_q:.4f}")
print("="*50)
print(f"\nvs classical baselines:")
for name, vals in baselines.items():
    print(f"  vs {name:<12}: acc {acc_q-vals['accuracy']:+.4f}  f1 {f1_q-vals['f1']:+.4f}")

---
## Step 5: Results table

In [ ]:
results_all = {name: {'accuracy': v['accuracy'], 'f1': v['f1']} for name, v in baselines.items()}
results_all['Quantum (IQP)'] = {'accuracy': acc_q, 'f1': f1_q}
print(f"{'Method':<16} {'Accuracy':>10} {'F1':>8}")
print("-"*36)
for name, v in results_all.items():
    mark = " <-- best F1" if v['f1']==max(r['f1'] for r in results_all.values()) else ""
    print(f"{name:<16} {v['accuracy']:>10.4f} {v['f1']:>8.4f}{mark}")

---
## Step 6: Comparison figure

In [ ]:
names = list(results_all.keys())
accs  = [results_all[n]['accuracy'] for n in names]
f1s   = [results_all[n]['f1'] for n in names]
colors = ['#5B8DB8','#5B8DB8','#5B8DB8','#D95F02']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, vals, ylab in [(axes[0], accs, 'Accuracy'), (axes[1], f1s, 'Macro F1')]:
    ax.bar(names, vals, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
    ax.set_ylim(0.4, 1.0); ax.set_title(f'{ylab} — AG News + MiniLM N=1000'); ax.set_ylabel(ylab)
    for i, v in enumerate(vals):
        ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11, fontweight='500')
    ax.set_xticklabels(names, rotation=15, ha='right')
blue = mpatches.Patch(color='#5B8DB8', label='Classical')
orange = mpatches.Patch(color='#D95F02', label='Quantum IQP')
fig.legend(handles=[blue, orange], loc='upper center', ncol=2, fontsize=11, bbox_to_anchor=(0.5,1.02))
plt.suptitle('Quantum vs Classical — AG News + MiniLM (8-dim, N=1000)', y=1.06)
plt.tight_layout(); plt.show()
save_figure(fig, f'{TAG}_fig1_comparison')

---
## Step 7: Learning curve

In [ ]:
train_sizes = [50, 100, 200, 400, 700, 1000]
lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}
print(f"{'N':>5}  {'Quantum':>9}  {'Linear':>9}  {'RBF':>9}")
print("-"*40)
for n in train_sizes:
    Kn, Kn_test = K_train[:n,:n], K_test[:,:n]
    yp = SVC(kernel='precomputed', C=1.0).fit(Kn, y_tr[:n]).predict(Kn_test)
    lc['Quantum (IQP)'].append(float(f1_score(y_te, yp, average='macro')))
    for cn, ck in [('Linear','linear'),('RBF','rbf')]:
        clf = SVC(kernel=ck, C=1.0, gamma='scale' if ck=='rbf' else 1.0,
                  random_state=42).fit(X_tr[:n], y_tr[:n])
        lc[cn].append(float(f1_score(y_te, clf.predict(X_te), average='macro')))
    print(f"{n:>5}  {lc['Quantum (IQP)'][-1]:>9.4f}  {lc['Linear'][-1]:>9.4f}  {lc['RBF'][-1]:>9.4f}")

fig2 = plt.figure(figsize=(8, 4))
cmap = {'Quantum (IQP)':'#D95F02','Linear':'#5B8DB8','RBF':'#2D9C4A'}
for m, vals in lc.items():
    ls = '-' if m=='Quantum (IQP)' else '--'
    lw = 2.5 if m=='Quantum (IQP)' else 1.8
    plt.plot(train_sizes, vals, label=m, color=cmap[m], linestyle=ls, linewidth=lw, marker='o', markersize=5)
plt.xlabel('Training set size (N)'); plt.ylabel('Macro F1')
plt.title('Learning curves — AG News + MiniLM\n8-dim embeddings')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
save_figure(fig2, f'{TAG}_fig2_learning_curves')

---
## Step 8: Save results JSON

In [ ]:
results_week3 = {
    'dataset': 'agnews', 'encoder': 'minilm',
    'n_classes': 4, 'metric': 'macro',
    'n_qubits': N_QUBITS, 'n_train': int(len(X_tr)), 'n_test': int(len(X_te)),
    'compute_time_min': round(elapsed/60, 1),
    'quantum_accuracy': acc_q, 'quantum_f1': f1_q,
    'all_results': results_all,
    'learning_curve_sizes': train_sizes,
    'learning_curve_results': {k: v for k, v in lc.items()},
}
checkpoint(f'{TAG}_results_week3.json',
           lambda p: json.dump(results_week3, open(p,'w'), indent=2))

print(f"\n=== WEEK 3 DONE — {TAG} ===")
print(f"Quantum IQP: acc={acc_q:.4f}  macro F1={f1_q:.4f}")
print(f"\nFiles for {TAG}:")
for f in sorted(os.listdir(SAVE_DIR)):
    if f.startswith(TAG) and 'week3' in f or (f.startswith(TAG) and 'kernel_matrices' in f) or (f.startswith(TAG) and 'fig1' in f) or (f.startswith(TAG) and 'fig2' in f):
        print(f"  {f}")

---
## Week 3 done — AG News + MiniLM

### Saved (tag = `agnews_minilm`):
- `agnews_minilm_kernel_matrices.npz` — saved immediately after compute (reused in Week 4)
- `agnews_minilm_fig1_comparison.png/.pdf`
- `agnews_minilm_fig2_learning_curves.png/.pdf`
- `agnews_minilm_results_week3.json`

### Next:
Week 4 for this config loads `agnews_minilm_kernel_matrices.npz` for the IQP 1L reuse.
Metric stays macro F1 throughout.